In [1]:
!pip install lancedb sentence-transformers pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 369.9/369.9 kB 11.3 MB/s eta 0:00:00


In [2]:
import lancedb
import pandas as pd
from sentence_transformers import SentenceTransformer

In [3]:
# 1. Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# 2. Connect to local LanceDB
db = lancedb.connect("./customer_support_lancedb")

# 3. Sample support documents
documents = [
    "If the router red light is blinking, restart the router and check the fiber cable.",
    "To reset your password, click Forgot Password on the login page.",
    "Warranty replacement is available within one year of purchase.",
    "If internet is slow, check Wi-Fi signal strength and restart the modem.",
    "For billing issues, contact accounts support with your invoice number."
]

ids = ["doc1", "doc2", "doc3", "doc4", "doc5"]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
# 4. Convert documents into embeddings
embeddings = model.encode(documents).tolist()

# 5. Create dataframe
data = pd.DataFrame({
    "id": ids,
    "text": documents,
    "vector": embeddings
})

# 6. Create or overwrite LanceDB table
table = db.create_table(
    "support_knowledge_base",
    data=data,
    mode="overwrite"
)

# 7. Customer question
question = "My router has a blinking red light. What should I do?"

# 8. Convert question into embedding
query_vector = model.encode(question).tolist()

# 9. Search LanceDB
results = table.search(query_vector).limit(2).to_pandas()

print("\nCustomer Question:")
print(question)

print("\nRelevant Knowledge Retrieved from LanceDB:")
for text in results["text"]:
    print("-", text)

print("\nFinal Answer:")
print(
    "A blinking red router light usually means there may be a connection issue. "
    "Please restart the router and check whether the fiber cable is properly connected. "
    "If the problem continues, contact technical support."
)


Customer Question:
My router has a blinking red light. What should I do?

Relevant Knowledge Retrieved from LanceDB:
- If the router red light is blinking, restart the router and check the fiber cable.
- If internet is slow, check Wi-Fi signal strength and restart the modem.

Final Answer:
A blinking red router light usually means there may be a connection issue. Please restart the router and check whether the fiber cable is properly connected. If the problem continues, contact technical support.
